# Kernel-Level Fine-Tuning with **Unsloth** — Custom Triton Kernels & Hand-Derived Gradients
### Tools & Frameworks  ·  Colab T4 (16 GB) ready

> Every other notebook in this repo optimises the **algorithm**: which loss, which data, which adapter. This one goes a layer down and optimises the **execution** — the same LoRA, the same cross-entropy, the same weights, producing the same gradients, in roughly half the time and a fraction of the memory.
>
> ⚠️ **One factual correction to the usual framing before we start.** Unsloth's kernels are **OpenAI Triton**, not hand-written C/CUDA — every file in `unsloth/kernels/` opens with `import triton, triton.language as tl`. And the larger half of the speedup is not the kernels at all: it is **hand-derived backward passes** written as `torch.autograd.Function`s, which change *what gets stored and recomputed*, not just how fast each op runs. Calling it "faster CUDA" undersells it and, worse, points you at the wrong lever if you ever need to do this yourself.
>
> Claims here are checked against `unslothai/unsloth@main` source, and the notebook **numerically verifies the fused gradients against PyTorch reference implementations** before benchmarking anything. A hand-derived gradient is code that can be *wrong* in ways autograd structurally cannot be — trusting it without a check is the single biggest risk of this approach.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)

- **Kernel fusion.** A GPU kernel launch reads its inputs from HBM and writes its outputs back. RMSNorm, RoPE, SwiGLU and the residual adds are **memory-bandwidth-bound**: their arithmetic intensity (FLOPs per byte moved) is `O(1)`. An A100-80GB moves ~2.0 TB/s across HBM while issuing tens of TFLOP/s of *non-tensor-core* fp16 arithmetic, so an op doing one FLOP per element read finishes computing while the memory system is still delivering — the kernel is a bandwidth job wearing a compute costume. Eager PyTorch executes each as its own kernel, so an `n`-op chain pays `n` round-trips. A fused Triton kernel pays **one read and one write** for the whole chain.
- **Custom `torch.autograd.Function` with an analytically derived backward.** Autograd is a *mechanical* differentiator: it stores every intermediate that any partial derivative might reference. A human who works the chain rule out on paper can see that most of them are reconstructible. `LoRA_MLP` in `unsloth/kernels/fast_lora.py` carries the derivation in its docstring —
  $$\frac{\partial C}{\partial G} = X^{\!\top}\big(D W^{\!\top}\odot \mathrm{d}f \odot g\big), \qquad \frac{\partial C}{\partial A_g} = \frac{\partial C}{\partial G} B^{\!\top}, \qquad \frac{\partial C}{\partial B_g} = A^{\!\top}\frac{\partial C}{\partial G}$$
  — and consequently stores only `X, e, g` for the entire gate/up/down SwiGLU block, recomputing the rest.
- **In-place gradient accumulation into dead buffers.** Two verified instances: `dX = torch.matmul(df, upW.t(), out = X)` writes the input gradient **over the saved activation `X`**, whose last read has already happened; and `Fast_CrossEntropyLoss.backward` returns `logits` itself after a Triton kernel has overwritten that buffer with `dlogits`. Autograd can never do this — it does not know a tensor is dead, so it must allocate.
- **Never materialising the logit softmax.** The single largest memory term in LLM fine-tuning is not the model. Unsloth's `_cross_entropy_forward` computes the row-wise `logsumexp` in a streaming pass (chunked at `MAX_FUSED_SIZE` when `vocab_size > 65,536`, which is why Gemma's 256 k vocab is a special case in the source) and the backward reconstructs `softmax − onehot` on the fly. **No full probability tensor and no separate gradient tensor ever exist.**
- **Activation offload checkpointing.** `use_gradient_checkpointing = "unsloth"` selects `Unsloth_Offloaded_Gradient_Checkpointer`, which asynchronously moves checkpointed layer inputs to **pinned system RAM** and streams them back during the backward — trading PCIe bandwidth (which is idle) for VRAM (which is the binding constraint).
- **Monkey-patching as the delivery mechanism.** `FastLanguageModel.from_pretrained` rebinds `LlamaRMSNorm.forward`, the attention RoPE application, the MLP forward, and `transformers`' `LOSS_MAPPING` entries to the fused implementations. This is why **`import unsloth` must run before `transformers` and `trl`** — and why the honest benchmark in Step 8 runs each arm in a **separate process**.

### One-sentence definition of the mechanics

> **Unsloth replaces the memory-bandwidth-bound and memory-hungry parts of a transformer's training step — RMSNorm, RoPE, SwiGLU/GeGLU, the LoRA-adapted projections, and cross-entropy — with fused Triton kernels wrapped in `torch.autograd.Function`s whose backward passes are derived by hand, so that fewer intermediate tensors are ever allocated and fewer bytes ever cross HBM, yielding numerically equivalent gradients at roughly 2× the throughput and a fraction of the VRAM.**

### The exact engineering problem it solves

- **The loss costs more memory than the model.** For Llama-3.2-1B (`vocab = 128,256`) at batch 2 × 1024 tokens, the logits alone are `2048 × 128,256 × 2 B = 525 MB` in fp16 — and the stock `transformers` path **upcasts to fp32** for numerical safety (1.05 GB), then allocates `dlogits` on top: `0.53 + 1.05 + 1.05 ≈` **2.6 GB of transient for a model whose 4-bit weights are ~0.8 GB — the loss costs 3× the model.** Scale to Llama-3.1-8B at 8 × 2048 and it is **~21 GB of transient loss memory**. This term scales with `batch × seq × vocab` and is completely independent of parameter count, which is why it dominates exactly when you most want longer contexts.
- **PEFT does not fix the activation problem.** QLoRA cuts optimiser and gradient memory, not activations and not logits. Once weights are 4-bit, activations *are* the memory profile — so the next win has to come from the execution layer.
- **Bandwidth, not FLOPs, is the real budget for half the step.** The normalisation/rotation/activation ops in a transformer contribute a small share of FLOPs and a large share of *time*. You cannot buy that back with a better GPU at the same bandwidth ratio; you buy it back by touching memory fewer times.
- **The alternative is writing this yourself.** Deriving, implementing and verifying a fused LoRA-MLP backward is a specialist's week and a permanent maintenance liability. The value of the library is that the derivation is done, tested, and kept in step with `transformers`.

---

### The Human Element — Hugging Face datasets, and why the dataset *is* the benchmark

This is a **performance** technique, so its dataset requirement is unusual: the corpus is not what you are studying, it is the **independent variable you must hold still.** Throughput is reported in tokens/sec and VRAM in GB — both are dominated by the **sequence-length distribution**, because padding ratio and the `batch × seq × vocab` logit tensor scale directly with it. Benchmark two frameworks on a bimodal corpus with different shuffles and you measure the shuffle.

| HF path | What it is | Why this shape, for this technique |
|---|---|---|
| **`yahma/alpaca-cleaned`**<br>`train`, 51,760 rows | The de-duplicated, error-corrected Alpaca set: `instruction` / `input` / `output`, single-turn. | **The control.** Short and, critically, **narrow** — the vast majority of examples land in a tight length band, so padding waste is nearly constant across batches and any throughput delta is attributable to the kernels rather than to batch composition. It is also the corpus behind most published Unsloth numbers, which makes results comparable. Used for both the training run and the benchmark below. |
| **`mlabonne/FineTome-100k`**<br>`train`, 100 k rows | Multi-turn ShareGPT-format conversations in a `conversations` column, filtered from The-Tome for instruction quality. | **The realistic case,** and the corpus in Unsloth's own current notebooks. Multi-turn means genuinely long sequences, which is where the RoPE and fused-CE kernels stop being a rounding error: the logit tensor grows linearly in sequence length while the 4-bit weights do not move at all. |
| **`HuggingFaceH4/ultrachat_200k`**<br>split `train_sft`, ~208 k rows | Large-scale multi-turn dialogue, the SFT stage of the Zephyr recipe. | **The stress case.** At 4–8 k contexts the baseline OOMs on hardware where the fused path still fits, which is the clearest possible demonstration that this is a *memory* technique before it is a *speed* technique. Also the honest place to measure, because it is what production SFT actually looks like. |

> **Methodological note that matters more than the dataset choice:** the benchmark in Step 8 does **not** feed raw examples to either arm. It tokenises `yahma/alpaca-cleaned` once, to a **fixed 1024-token block length**, and hands both arms the identical tensor — so `tokens_forwarded` is equal by construction and tokens/sec is a real ratio rather than an artifact of who padded less.

---

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the engineering and mathematical reason for this implementation

**1. Bandwidth-bound ops are where eager PyTorch leaks time.**
A decoder layer's RMSNorm → RoPE → attention → RMSNorm → SwiGLU chain contains a handful of huge GEMMs (compute-bound, already near peak) and a long tail of elementwise/reduction ops (bandwidth-bound). For the tail, time ≈ `bytes_moved / HBM_bandwidth`, and eager execution multiplies `bytes_moved` by the number of ops in the chain. `unsloth/kernels/rms_layernorm.py` and `swiglu.py` collapse each chain into one kernel: one read, one write. There is no algorithmic cleverness here — just the observation that the arithmetic was never the bottleneck.

**2. The RoPE kernel derives its own backward from a symmetry.**
`rope_embedding.py` compiles **one** Triton kernel with `BACKWARD_PASS: tl.constexpr` and flips the sign of the sine term. RoPE is an orthogonal rotation by θ, so its Jacobian-vector product is a rotation by −θ — the backward is the forward with `sin → −sin`, and neither direction needs to save anything. Autograd would have saved `cos` and `sin` per layer per step.

**3. The hand-derived LoRA-MLP backward stores three tensors instead of ten.**
For SwiGLU with LoRA on all three projections, the forward computes `e = XG`, `g = XU`, `h = silu(e)·g`, `i = hW`, plus a low-rank pair `XA, (XA)B` for each of the three. Autograd conservatively saves nearly all of them. Unsloth's `LoRA_MLP.forward` saves `gateA, gateB, upA, upB, downA, downB, X, e, g` — the adapters (tiny, `r×d`) and **three** activations — and reconstructs `h`, `df` and `de` inside the backward from the identity `df = σ(e)(1−f) + f`. Recomputation is nearly free here precisely because these ops are bandwidth-bound; the *storage* was the expensive part.

**4. Dead buffers are reused for gradients.** Two verified places where the code does something autograd is structurally forbidden from doing:
- `dX = torch.matmul(df, upW.t(), out = X if ctx.inplace else None)` — the input gradient is written **into the saved activation's storage**, because `X`'s last read has already occurred.
- `Fast_CrossEntropyLoss.backward` runs a Triton kernel over `logits` and then `return logits, None, None, None` — the logits buffer **is** the gradient buffer.

**5. Cross-entropy never materialises a probability tensor.** The forward emits only `losses` (n_rows floats) and `logsumexp` (n_rows floats); the backward reconstructs `softmax − onehot` per block on the fly. Where stock `transformers` peaks at roughly `3 ×` a `(batch·seq, vocab)` fp32 tensor, the fused path peaks at the fp16 logits it was already given, plus `2 × n_rows` floats.

#### VRAM & Compute Impact

The loss-memory arithmetic, which is the term people forget:

| Config | `batch·seq` | vocab | logits fp16 | stock peak (fp32 logits + softmax + dlogits) | fused CE peak |
|---|---|---|---|---|---|
| Llama-3.2-1B, 2×1024 | 2,048 | 128,256 | 0.53 GB | **~2.6 GB** | ~0.53 GB |
| Llama-3.1-8B, 8×2048 | 16,384 | 128,256 | 4.2 GB | **~21 GB** | ~4.2 GB |
| Gemma-2-9B, 4×2048 | 8,192 | 256,000 | 4.2 GB | **~21 GB** | ~4.2 GB |

Where the rest of the budget goes, and what each mechanism buys:

| Mechanism | Buys | Costs |
|---|---|---|
| Fused CE (no softmax materialisation) | the table above — the single biggest line | nothing measurable |
| Fused RMSNorm / RoPE / SwiGLU | ~1 read+write instead of ~n per chain | Triton JIT warmup on first step |
| Hand-derived LoRA-MLP/QKV backward | fewer saved activations; fewer kernel launches | recompute of `h, df, de` (bandwidth-bound, cheap) |
| In-place `dX` / `dlogits` | one fewer full-size allocation each | the buffers are destroyed — see the correctness cell |
| `use_gradient_checkpointing = "unsloth"` | activations move to pinned host RAM ⇒ much longer contexts | PCIe traffic; ~a few % step time |

- **FLOPs are unchanged.** This is not an approximation and not a smaller model — the same `6·N·D` arithmetic happens, with the same numerics. What changes is bytes moved and bytes held.
- **The speedup is configuration-dependent, and honesty about that matters.** Gains are largest when the loss and the bandwidth-bound tail are a large share of the step — small model, large vocab, long sequence, small batch. Gains are smallest when you are already GEMM-bound. Upstream reports **~2× faster and up to 70 % less VRAM** for standard LoRA/QLoRA SFT, ~12× for MoE, and ~30 % less VRAM with the newer packing path; treat any single number as a claim about a *configuration*, which is exactly why Step 8 measures your own.
- **On a T4 specifically:** `sm_75`, fp16 only (no bf16), no FlashAttention-2. Unsloth still applies its Triton kernels and its own attention path, so the technique demonstrates fine — but the absolute numbers are lower than an Ampere run, and the CE win is proportionally *larger* because there is less GEMM throughput to hide behind.

#### Pros & Cons

**Pros**
- **The one optimisation with no modelling trade-off.** Same weights, same objective, same gradients — you are not quantising further, not lowering rank, not truncating context.
- **Memory headroom converts directly into capability**: longer sequences, bigger batches, or a larger model on the same card.
- **Drop-in at the call site.** `FastLanguageModel.from_pretrained` + `get_peft_model`, then your existing TRL `SFTTrainer` code is unchanged.
- **Export is handled**: `save_pretrained_merged` (16-bit / 4-bit), `save_pretrained_gguf` with the full llama.cpp quant matrix (`q4_k_m`, `q5_k_m`, `q8_0`, …), and vLLM-compatible compressed-tensors exports.
- **It is a legible reference implementation.** The `LoRA_MLP` docstring contains the actual derivation; this is the best-documented worked example of custom transformer autograd in open source.

**Cons**
- **Global monkey-patching.** Patches land on `transformers` classes at import. `import unsloth` must come first; anything else in the process is affected; and **a before/after benchmark inside one process is invalid** — the "baseline" is already patched. Step 8 uses subprocesses for exactly this reason.
- **Correctness is now your problem to verify.** A hand-derived backward can be subtly wrong (a missing scale, a dtype cast) in a way that trains to a slightly worse model without ever raising. Verify numerically. This notebook does.
- **Destructive in-place semantics.** The fused CE overwrites its input logits buffer during backward. Fine inside a training step, a trap if you were holding a reference to those logits.
- **Architecture allow-list.** Fused paths exist for the supported families (Llama, Mistral, Gemma, Qwen, Cohere, Granite, Falcon-H1, MoE variants…). A model outside the list silently falls back to slower paths, or fails.
- **Tight version coupling.** Patching `transformers` internals means a `transformers` release can break it; pin both together.
- **Multi-GPU is the weakest area.** Upstream states multi-GPU is "available now, with a major upgrade on the way" — the flagship benchmarks, and this notebook, are single-GPU. Do not assume the 2× transfers to your 8-GPU DDP job without measuring it.
- **Mixed licensing.** Verified in the file headers: `kernels/fast_lora.py` and `kernels/cross_entropy_loss.py` are Apache-2.0, while `kernels/rope_embedding.py` carries **LGPL-3.0**. Read the header of the file you actually depend on, and route redistribution plans past whoever owns that decision.

#### Metrics to watch

- **Peak VRAM — `max_memory_allocated` *and* `max_memory_reserved`.** Allocated is what the tensors need; reserved is what the caching allocator holds and therefore what actually OOMs you. Reporting only one is how benchmarks mislead.
- **Tokens/sec, computed from an identical token budget on both arms.** Not `it/s` — iterations hide different batch shapes.
- **Median step time, warmup excluded.** Triton JIT-compiles on first use; including step 0 can flatter or penalise either arm by seconds.
- **Loss parity.** Run both arms a few steps and compare the loss curve. Equivalent kernels produce near-identical losses (fp16 noise only). Divergence is a correctness bug, not a speedup.
- **Max absolute gradient error vs. the reference implementation** — the check in Step 3. For fp16 inputs, `~1e-3` is fp16 rounding; `~1e-1` is a bug.
- **Sequence length at which the baseline OOMs but the fused path does not** — the most honest single-number summary of what this buys.

---

## 3. Production-Grade Implementation (Colab T4, 16 GB)

**`unsloth/Llama-3.2-1B-Instruct` + QLoRA on `yahma/alpaca-cleaned`, with the fused kernels verified against PyTorch reference implementations and benchmarked head-to-head against a stock `transformers` + `peft` + `trl` baseline.**

> ⚙️ **Why a 1B model with a 128 k vocab is the right demonstrator, not a cop-out.** The fused-CE argument scales with `batch × seq × vocab` and is *independent of parameter count*. Llama-3.2-1B has a **128,256-token vocabulary attached to 1.2 B parameters** — the most lopsided ratio in common use — so the logit tensor is several times the size of the 4-bit weights. Everything the kernels do is visible at T4 scale; only the absolute numbers change on an H100.

**Executable pipeline:**

| Step | What | The point |
|---|---|---|
| 0 | Install + **import-order guard** | `import unsloth` before `transformers`, or the patches never land |
| 1 | `FastLanguageModel.from_pretrained` + **prove the patch happened** | patched classes are introspectable; assert, don't assume |
| 2 | `FastLanguageModel.get_peft_model` | LoRA through the fused layer-wrapping path |
| 3 | **Numerical equivalence check** of fused CE / RMSNorm / SwiGLU vs. PyTorch | the cell nobody writes and everybody needs |
| 4 | `yahma/alpaca-cleaned` → chat-templated text | the control corpus |
| 5 | Real `SFTTrainer` run + VRAM/throughput readout | it has to actually train |
| 6 | Export: merged 16-bit + GGUF `q4_k_m` | the deployable artifact |
| 7 | Inference via `FastLanguageModel.for_inference` | 2× faster generation path |
| 8 | **Head-to-head benchmark in two subprocesses** + plot | the only methodologically valid way to measure it |

### Environment Setup

In [ ]:
# Colab ships torch/CUDA already; this adds Unsloth and its zoo of patches.
# Locally, the upstream-recommended form is:  uv pip install unsloth --torch-backend=auto
%pip install -q --upgrade unsloth unsloth_zoo
%pip install -q --upgrade "trl>=0.19" "peft>=0.13" "transformers>=4.45" bitsandbytes datasets matplotlib

In [ ]:
# ==========================================================================================
# IMPORT ORDER IS LOAD-BEARING.
# Unsloth delivers its kernels by monkey-patching transformers classes at import time
# (LlamaRMSNorm.forward, the RoPE application, the MLP forward, transformers' LOSS_MAPPING).
# If transformers/trl are imported first, some of those bindings are already resolved and
# the patch is a no-op -- you get the slow path with none of the errors.
# ==========================================================================================
import sys

_already = [m for m in ("transformers", "trl", "peft") if m in sys.modules]
if _already:
    print(f"WARNING: {_already} imported before unsloth — restart the runtime and run this cell first.")

import unsloth  # noqa: F401  <- MUST be the first heavy import
from unsloth import FastLanguageModel

import gc
import json
import os
import time
from pathlib import Path

import torch

# ------------------------------------------------------------------ hardware probe
HAS_CUDA = torch.cuda.is_available()
CC = torch.cuda.get_device_capability() if HAS_CUDA else (0, 0)
SM = CC[0] * 10 + CC[1]
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else "CPU"
SUPPORTS_BF16 = SM >= 80          # bf16 and FlashAttention-2 both need Ampere
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

import triton
import transformers

print(f"gpu          : {GPU_NAME}  (sm_{SM})")
print(f"torch        : {torch.__version__}")
print(f"triton       : {triton.__version__}        <- the kernels are Triton, not C/CUDA")
print(f"transformers : {transformers.__version__}")
print(f"unsloth      : {getattr(unsloth, '__version__', 'unknown')}")
print(f"dtype        : {DTYPE}   (bf16 requires sm_80+)")

WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
os.chdir(WORKDIR)
print(f"workdir      : {WORKDIR}")

### Step 1 — Load the model, then **prove the kernels were actually injected**

`FastLanguageModel.from_pretrained` does three things at once: loads the checkpoint (optionally pre-quantised to 4-bit NF4), rebinds the model's normalisation / RoPE / MLP / loss functions to the fused Triton implementations, and installs the offloaded gradient checkpointer.

The second of those is **silent**. If the model family is outside the allow-list, or the import order was wrong, or a `transformers` release moved a symbol, you get a working model on the slow path and no error. So the cell below does not take the patch on faith — it **inspects the live module objects** and reports which classes are Unsloth's.

`use_gradient_checkpointing = "unsloth"` is not a boolean here: it selects `Unsloth_Offloaded_Gradient_Checkpointer`, which moves checkpointed activations to pinned host RAM instead of holding them in VRAM.

In [ ]:
MAX_SEQ_LEN = 1024
BASE_MODEL = "unsloth/Llama-3.2-1B-Instruct"   # 1.2B params, 128,256-token vocab

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,   # None -> auto-select (fp16 on a T4, bf16 on Ampere+)
    load_in_4bit = True,   # NF4 QLoRA; set False for 16-bit LoRA
    use_gradient_checkpointing = "unsloth",   # NOT True: selects the host-RAM offloading checkpointer
)

# ---- Verify the patch, do not assume it -------------------------------------------------
inner = model.model if hasattr(model, "model") else model
layer0 = inner.model.layers[0] if hasattr(inner, "model") else inner.layers[0]

checks = {
    "input_layernorm": type(layer0.input_layernorm).__name__,
    "mlp": type(layer0.mlp).__name__,
    "mlp.forward": getattr(layer0.mlp.forward, "__qualname__", "?"),
    "self_attn.forward": getattr(layer0.self_attn.forward, "__qualname__", "?"),
    "loss_function": getattr(getattr(inner, "loss_function", None), "__name__", "?"),
}
print(f"{'component':<20}{'bound implementation':<46}patched?")
print("-" * 78)
for k, v in checks.items():
    hit = "unsloth" in v.lower() or "fast" in v.lower()
    print(f"{k:<20}{v:<46}{'YES' if hit else 'check'}")

# The fused kernels themselves — importable, inspectable, and used directly in Step 3.
from unsloth.kernels import (
    fast_cross_entropy_loss,   # streaming logsumexp; overwrites logits with dlogits
    fast_rms_layernorm,   # fused normalise + scale
    swiglu_fg_kernel,   # silu(e) * g in ONE kernel
    fast_rope_embedding,   # one kernel, BACKWARD_PASS flips sin's sign
)
print("\nfused kernels imported:", [f.__name__ for f in
      (fast_cross_entropy_loss, fast_rms_layernorm, swiglu_fg_kernel, fast_rope_embedding)])

mem_after_load = torch.cuda.max_memory_allocated() / 1e9
print(f"\nVRAM after load: {mem_after_load:.2f} GB")

### Step 2 — LoRA through Unsloth's layer-wrapping path

The signature mirrors `peft.LoraConfig`, but the wrapping is different: targeting the MLP trio (`gate_proj`, `up_proj`, `down_proj`) routes the whole block through `LoRA_MLP` — the single `autograd.Function` whose derivation is in Section 2 — instead of three independently-differentiated PEFT layers. Targeting `q/k/v/o` likewise routes through `apply_lora_qkv` / `apply_lora_o`.

That is why the target-module list is a **performance** decision here and not only a capacity one: dropping the MLP trio does not just reduce rank coverage, it opts you out of the fused MLP backward entirely.

`lora_dropout = 0` and `bias = "none"` are load-bearing for the same reason — both are on the fused fast path; non-zero dropout or trainable biases fall back to the generic implementation.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # -> apply_lora_qkv / apply_lora_o
        "gate_proj", "up_proj", "down_proj",      # -> LoRA_MLP (the fused SwiGLU backward)
    ],
    lora_alpha = 16,
    lora_dropout = 0,  # 0 keeps the fused path; any dropout falls back to generic PEFT
    bias = "none",   # "none" keeps the fused path
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,   # rank-stabilised scaling: alpha/sqrt(r) instead of alpha/r
    loftq_config = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M  ({100*trainable/total:.3f}%)")
print(f"VRAM now : {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

### Step 3 — Numerical equivalence: the cell nobody writes

A hand-derived backward is **code**, and code can be wrong. Autograd cannot silently return a wrong gradient for a correctly-written forward; a manual `backward()` absolutely can — a dropped scale factor, a missed dtype cast, a transposed operand. The failure mode is not a crash, it is a model that trains slightly worse forever.

So before benchmarking anything, check the fused kernels against PyTorch reference implementations on **both value and gradient**:

- **Cross-entropy** — the highest-value and highest-risk kernel. Note the `.clone()` on each input: `Fast_CrossEntropyLoss.backward` **overwrites its logits buffer with `dlogits`**, so reusing the tensor afterwards would silently compare a gradient against a gradient.
- **RMSNorm** and **SwiGLU** — the fused elementwise chains, checked on the model's own live `LlamaRMSNorm` module so the weights and epsilon are real.

**Reading the numbers:** inputs are fp16, whose machine epsilon is ~`9.8e-4`. Errors at `1e-3`–`1e-2` on a value of order 1 are rounding. Errors at `1e-1` or a different *sign pattern* are a bug — stop and investigate rather than shipping a fast wrong model.

In [ ]:
import torch.nn.functional as F

torch.manual_seed(0)
dev = model.device
V = model.config.vocab_size
N, D = 512, model.config.hidden_size

print(f"vocab={V:,}  hidden={D}  rows={N}\n")

# ---- 1. Fused cross-entropy: value AND gradient -----------------------------------------
logits_ref = (torch.randn(1, N, V, device=dev, dtype=torch.float16) * 2).requires_grad_(True)
labels = torch.randint(0, V, (1, N), device=dev)
labels[:, ::7] = -100   # exercise the ignore_index path

# IMPORTANT: clone. The fused backward writes dlogits INTO this buffer.
logits_fast = logits_ref.detach().clone().requires_grad_(True)

loss_ref = F.cross_entropy(
    logits_ref.float().view(-1, V), labels.view(-1), ignore_index=-100, reduction="mean"
)
loss_fast = fast_cross_entropy_loss(logits_fast, labels)

g_ref, = torch.autograd.grad(loss_ref, logits_ref)
g_fast, = torch.autograd.grad(loss_fast, logits_fast)

print(f"{'check':<34}{'reference':>13}{'fused':>13}{'max |Δ|':>13}")
print("-" * 73)
print(f"{'cross_entropy value':<34}{loss_ref.item():>13.6f}{loss_fast.item():>13.6f}"
      f"{abs(loss_ref.item()-loss_fast.item()):>13.2e}")
print(f"{'cross_entropy dlogits':<34}{'':>13}{'':>13}"
      f"{(g_ref.float()-g_fast.float()).abs().max().item():>13.2e}")

# ---- 2. Fused RMSNorm against the model's own live module --------------------------------
rms = layer0.input_layernorm
x_ref = torch.randn(1, N, D, device=dev, dtype=torch.float16, requires_grad=True)
x_fast = x_ref.detach().clone().requires_grad_(True)

def rms_reference(x, w, eps):
    # The textbook definition, computed in fp32 exactly as transformers does it, then cast
    # back to x's dtype. The cast matters: Unsloth upcasts norm weights to fp32, so without
    # it you would be comparing fp32 against fp16 and calling the dtype gap a kernel bug.
    v = x.float().pow(2).mean(-1, keepdim=True)
    return ((x.float() * torch.rsqrt(v + eps)) * w.float()).to(x.dtype)

eps = getattr(rms, "variance_epsilon", getattr(rms, "eps", 1e-5))
y_ref = rms_reference(x_ref, rms.weight, eps)
y_fast = fast_rms_layernorm(rms, x_fast)
y_ref.sum().backward(); y_fast.sum().backward()

print(f"{'rms_layernorm forward':<34}{'':>13}{'':>13}"
      f"{(y_ref-y_fast).abs().max().item():>13.2e}")
print(f"{'rms_layernorm dX':<34}{'':>13}{'':>13}"
      f"{(x_ref.grad-x_fast.grad).abs().max().item():>13.2e}")

# ---- 3. Fused SwiGLU: h = silu(e) * g in one kernel instead of three ---------------------
e = torch.randn(1, N, D, device=dev, dtype=torch.float16)
g = torch.randn(1, N, D, device=dev, dtype=torch.float16)
h_ref = F.silu(e.float()).to(e.dtype) * g
h_fast = swiglu_fg_kernel(e, g)
print(f"{'swiglu forward':<34}{'':>13}{'':>13}"
      f"{(h_ref-h_fast).abs().max().item():>13.2e}")

print("\nfp16 eps = 9.77e-04. Deltas at 1e-3..1e-2 are rounding; 1e-1 is a bug.")
del logits_ref, logits_fast, g_ref, g_fast, x_ref, x_fast, y_ref, y_fast, e, g, h_ref, h_fast
gc.collect(); torch.cuda.empty_cache()

### Step 4 — The corpus

`yahma/alpaca-cleaned` rendered through the model's own chat template. Nothing here is Unsloth-specific — that is the point: the kernels sit under an ordinary TRL training loop, and the data path is unchanged from any other SFT notebook in this repo.

One detail worth keeping: **`EOS` must be appended**, or generation never terminates. `SFTTrainer` will not add it for you when you supply a pre-rendered text field.

In [ ]:
from datasets import load_dataset

alpaca = load_dataset("yahma/alpaca-cleaned", split="train")
print(alpaca)

EOS = tokenizer.eos_token

def to_text(ex):
    user = ex["instruction"] if not ex["input"] else f"{ex['instruction']}\n\n{ex['input']}"
    messages = [
        {"role": "user", "content": user},
        {"role": "assistant", "content": ex["output"]},
    ]
    # tokenize=False renders the template to a string; SFTTrainer tokenizes it later.
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text if text.rstrip().endswith(EOS) else text + EOS}

train_ds = alpaca.shuffle(seed=3407).select(range(2000)).map(
    to_text, remove_columns=alpaca.column_names
)
print("\n" + train_ds[0]["text"][:400])

lens = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in train_ds["text"][:500]]
lens.sort()
print(f"\ntoken lengths  p50={lens[len(lens)//2]}  p90={lens[int(.9*len(lens))]}  max={lens[-1]}")
print("narrow distribution => padding waste is near-constant => a clean throughput A/B")

### Step 5 — Train

Ordinary `trl.SFTTrainer`. The kernels are already inside the model object; nothing in this cell knows they exist. That is the whole ergonomic argument for the library.

`fp16` vs `bf16` is selected from the probe — a T4 has no bf16 tensor cores, and forcing it produces either an error or a silent emulated slowdown.

In [ ]:
from trl import SFTConfig, SFTTrainer

torch.cuda.reset_peak_memory_stats()

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,   # `tokenizer=` is deprecated in recent TRL
    train_dataset = train_ds,
    args = SFTConfig(
        output_dir = "./outputs/unsloth-llama32-1b",
        dataset_text_field = "text",
        max_length = MAX_SEQ_LEN,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch = 8 sequences
        max_steps = 60,   # a demonstration run, not a converged model
        warmup_steps = 5,
        learning_rate = 2e-4,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",   # 8-bit moments: ~4x less optimiser state
        weight_decay = 0.01,
        logging_steps = 5,
        fp16 = not SUPPORTS_BF16,   # T4: fp16 only
        bf16 = SUPPORTS_BF16,
        seed = 3407,
        report_to = "none",
    ),
)

t0 = time.time()
stats = trainer.train()
wall = time.time() - t0

peak = torch.cuda.max_memory_allocated() / 1e9
reserved = torch.cuda.max_memory_reserved() / 1e9
tokens = 60 * 2 * 4 * MAX_SEQ_LEN   # steps * micro_bs * accum * seq (upper bound)

print(f"\nwall clock: {wall/60:.2f} min")
print(f"peak allocated: {peak:.2f} GB")
print(f"peak reserved: {reserved:.2f} GB   <- this is what actually OOMs you")
print(f"final train_loss: {stats.training_loss:.4f}")
print(f"~throughput: {tokens/wall:,.0f} tokens/sec (padded upper bound)")

### Step 6 — Export

Three shapes, three audiences:

| Call | Produces | For |
|---|---|---|
| `model.save_pretrained(dir)` | LoRA adapter only (~50 MB) | stacking another stage, or shipping a delta |
| `save_pretrained_merged(dir, tok, save_method="merged_16bit")` | dense fp16 checkpoint | vLLM / TGI / anything that speaks `transformers` |
| `save_pretrained_gguf(dir, tok, quantization_method="q4_k_m")` | GGUF | llama.cpp / Ollama / LM Studio — CPU and consumer GPUs |

`q4_k_m` is the standard quality/size default (Q6_K for half the `attention.wv` and `feed_forward.w2` tensors, Q4_K elsewhere); `q8_0` and `q5_k_m` trade size for fidelity, and the full matrix — including imatrix-requiring `iq*` quants — is enumerated in `unsloth/save.py`.

> ⚠️ **GGUF export builds llama.cpp from source on first use** — several minutes on Colab, and it needs disk. It is behind a flag so the notebook stays runnable; flip it when you actually want the artifact.

In [ ]:
EXPORT_MERGED_16BIT = True
EXPORT_GGUF = False   # compiles llama.cpp on first run — slow on Colab

# 1. Adapter only — always cheap, always worth saving.
model.save_pretrained("./outputs/lora_adapter")
tokenizer.save_pretrained("./outputs/lora_adapter")
adapter_mb = sum(p.stat().st_size for p in Path("./outputs/lora_adapter").rglob("*")) / 1e6
print(f"adapter: {adapter_mb:.1f} MB")

# 2. Merged 16-bit — dequantises the 4-bit base and folds the adapter in.
#    NOTE this is lossy relative to (4-bit base + adapter): the merge happens in bf16/fp16,
#    so merged weights are not bit-identical to what you just evaluated. Merge to SERVE,
#    keep the adapter to EVALUATE.
if EXPORT_MERGED_16BIT:
    model.save_pretrained_merged("./outputs/merged_16bit", tokenizer, save_method="merged_16bit")
    merged_gb = sum(p.stat().st_size for p in Path("./outputs/merged_16bit").rglob("*")) / 1e9
    print(f"merged 16-bit: {merged_gb:.2f} GB")

if EXPORT_GGUF:
    model.save_pretrained_gguf("./outputs/gguf", tokenizer, quantization_method="q4_k_m")
    for p in Path("./outputs/gguf").rglob("*.gguf"):
        print(f"gguf: {p.name}  {p.stat().st_size/1e9:.2f} GB")

### Step 7 — Inference

`FastLanguageModel.for_inference` switches the model into Unsloth's generation path (fused kernels, no training-only bookkeeping). Call `for_training` before resuming training — the two modes are not interchangeable.

In [ ]:
FastLanguageModel.for_inference(model)      # 2x faster generation path

messages = [{"role": "user",
             "content": "Explain what a fused CUDA kernel is, in two sentences."}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

t0 = time.time()
out = model.generate(
    input_ids=inputs, max_new_tokens=128,
    do_sample=True, temperature=0.7, top_p=0.9,
    pad_token_id=tokenizer.eos_token_id,
)
dt = time.time() - t0
new_tokens = out.shape[1] - inputs.shape[1]

print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
print(f"\n{new_tokens} tokens in {dt:.2f}s = {new_tokens/dt:.1f} tok/s")

# FastLanguageModel.for_training(model)   # <- required before any further trainer.train()

---

### Step 8 — The head-to-head benchmark, and **why it must run in subprocesses**

This is the most important methodological point in the notebook, and it is the reason most published framework comparisons are worthless.

**`import unsloth` patches `transformers` globally and irreversibly for the life of the process.** `LlamaRMSNorm.forward`, the RoPE application, the MLP forward and `transformers.loss.loss_utils.LOSS_MAPPING` are all rebound at import. So a notebook that imports Unsloth, measures it, then builds a "stock baseline" with `AutoModelForCausalLM` **is measuring Unsloth against Unsloth** — the baseline is already patched, the delta collapses toward zero, and the conclusion is "the kernels barely help." Deleting the model, calling `empty_cache()`, even `del sys.modules["unsloth"]` do not undo a rebound method.

The fix is a process boundary. Each arm gets a fresh interpreter that imports exactly what it needs. This also gives clean VRAM accounting for free, since process exit releases everything the caching allocator held.

**What is held constant** (fairness is a design property, not an afterthought):

- **Identical source weights.** Both arms load the *same repo* and quantise it themselves. The Unsloth arm passes `use_exact_model_name=True` to suppress the automatic redirect to a pre-quantised mirror, so neither arm gets a head start from someone else's quantisation.
- **Identical token budget.** The dataset is tokenised once to fixed 1024-token blocks, so `steps × batch × 1024` tokens are forwarded by both arms. Tokens/sec is then a real ratio, not a padding artifact.
- **Identical LoRA config, optimiser, LR, seed, batch shape, and step count.**
- **Warmup excluded.** Triton JIT-compiles on first use; timing starts after warmup steps and peak-memory stats are reset at the same boundary, so we measure steady state.

**What is deliberately *not* held constant, and why:** the Unsloth arm uses `use_gradient_checkpointing="unsloth"` (host-RAM offload) while the baseline uses stock `gradient_checkpointing=True`. That *is* part of what the library ships, so leaving it in measures the real-world delta. To isolate kernel gains from offload gains, re-run the Unsloth arm with `--ckpt true` — the ablation is one flag.

In [ ]:
bench_py = r"""
import argparse, json, time, os, sys

ap = argparse.ArgumentParser()
ap.add_argument("--arm", choices=["unsloth", "hf"], required=True)
ap.add_argument("--ckpt", default="unsloth", help="unsloth | true  (gradient checkpointing mode)")
ap.add_argument("--steps", type=int, default=20)
ap.add_argument("--warmup", type=int, default=3)
ap.add_argument("--bs", type=int, default=2)
ap.add_argument("--seq", type=int, default=1024)
args = ap.parse_args()

MODEL = "unsloth/Llama-3.2-1B-Instruct"
SEED = 3407

# --- ARM-SPECIFIC IMPORTS ------------------------------------------------------------
# The unsloth arm imports unsloth FIRST (patching transformers). The hf arm never imports
# it at all, so its transformers is pristine. This separation is the entire point of
# running in subprocesses.
if args.arm == "unsloth":
    from unsloth import FastLanguageModel

import torch
from datasets import Dataset
# transformers.Trainer + default_data_collator, NOT SFTTrainer: the data is already
# tokenised to fixed-length blocks, and a plain CLM Trainer removes every version-dependent
# TRL preprocessing step that could differ between the two arms. Fewer moving parts = a
# comparison that measures kernels.
from transformers import (
    AutoTokenizer,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    default_data_collator,
)

torch.manual_seed(SEED)
SUPPORTS_BF16 = torch.cuda.get_device_capability()[0] >= 8

# --- IDENTICAL DATA: fixed-length blocks, so both arms forward the SAME token count ----
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

from datasets import load_dataset
raw = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=SEED)
ids = []
for row in raw:
    user = row["instruction"] if not row["input"] else row["instruction"] + "\n\n" + row["input"]
    text = tok.apply_chat_template(
        [{"role": "user", "content": user}, {"role": "assistant", "content": row["output"]}],
        tokenize=False,
    )
    ids.extend(tok(text, add_special_tokens=False)["input_ids"] + [tok.eos_token_id])
    if len(ids) >= args.seq * (args.steps + args.warmup + 2) * args.bs:
        break
n_blocks = len(ids) // args.seq
blocks = [ids[i * args.seq:(i + 1) * args.seq] for i in range(n_blocks)]
ds = Dataset.from_dict({"input_ids": blocks, "labels": [b[:] for b in blocks],
                        "attention_mask": [[1] * args.seq for _ in blocks]})

# --- MODEL ---------------------------------------------------------------------------
LORA = dict(r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"])

if args.arm == "unsloth":
    model, _ = FastLanguageModel.from_pretrained(
        model_name=MODEL, max_seq_length=args.seq, dtype=None, load_in_4bit=True,
        use_gradient_checkpointing=args.ckpt if args.ckpt != "true" else True,
        use_exact_model_name=True,   # do NOT redirect to a pre-quantised mirror: same weights both arms
    )
    model = FastLanguageModel.get_peft_model(
        model, use_gradient_checkpointing=args.ckpt if args.ckpt != "true" else True,
        random_state=SEED, **LORA,
    )
    hf_grad_ckpt = False   # already installed by unsloth
else:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if SUPPORTS_BF16 else torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, quantization_config=bnb, device_map="auto",
        attn_implementation="sdpa",   # the fair non-flash baseline; T4 cannot run FA2 anyway
    )
    # use_gradient_checkpointing=False here: let the Trainer own checkpointing so there is
    # exactly one owner and the reentrant setting is unambiguous.
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
    model = get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", **LORA))
    hf_grad_ckpt = True

torch.cuda.synchronize()
mem_load = torch.cuda.max_memory_allocated() / 1e9

# --- TIMING: exclude warmup, reset peak stats at the warmup boundary -------------------
class Stopwatch(TrainerCallback):
    def __init__(self, warmup):
        self.warmup, self.times, self.t = warmup, [], None
    def on_step_end(self, cfg, state, control, **kw):
        torch.cuda.synchronize()
        now = time.perf_counter()
        if state.global_step == self.warmup:
            torch.cuda.reset_peak_memory_stats()   # measure STEADY STATE, not the load spike
            self.t = now
        elif state.global_step > self.warmup:
            self.times.append(now - self.t)
            self.t = now

watch = Stopwatch(args.warmup)
trainer = Trainer(
    model=model,
    train_dataset=ds,
    data_collator=default_data_collator,   # rows are already exactly `seq` long: zero padding
    callbacks=[watch],
    args=TrainingArguments(
        output_dir=f"./bench_{args.arm}",
        per_device_train_batch_size=args.bs, gradient_accumulation_steps=1,
        max_steps=args.steps + args.warmup, warmup_steps=0, learning_rate=2e-4,
        optim="adamw_8bit", logging_steps=1000, save_strategy="no",
        fp16=not SUPPORTS_BF16, bf16=SUPPORTS_BF16, seed=SEED, report_to="none",
        gradient_checkpointing=hf_grad_ckpt,
        gradient_checkpointing_kwargs={"use_reentrant": False} if hf_grad_ckpt else None,
        remove_unused_columns=False,
    ),
)
out = trainer.train()

times = sorted(watch.times)
median = times[len(times) // 2] if times else float("nan")
print("UNSLOTH_BENCH_JSON " + json.dumps({
    "arm": args.arm,
    "ckpt": args.ckpt,
    "median_step_s": median,
    "tokens_per_sec": args.bs * args.seq / median if median == median else None,
    "peak_alloc_gb": torch.cuda.max_memory_allocated() / 1e9,
    "peak_reserved_gb": torch.cuda.max_memory_reserved() / 1e9,
    "mem_after_load_gb": mem_load,
    "final_loss": out.training_loss,
    "steps_timed": len(times),
    "tokens_forwarded": args.bs * args.seq * args.steps,
}))
"""

Path("bench_arm.py").write_text(bench_py, encoding="utf-8")
print(f"wrote bench_arm.py ({len(bench_py.splitlines())} lines)")

In [ ]:
import re
import subprocess

def run_arm(arm, **kw):
    """Fresh interpreter per arm — the only way the 'stock' baseline is genuinely stock."""
    cmd = [sys.executable, "bench_arm.py", "--arm", arm]
    for k, v in kw.items():
        cmd += [f"--{k}", str(v)]
    print(f"$ {' '.join(cmd)}")
    p = subprocess.run(cmd, capture_output=True, text=True)
    m = re.search(r"UNSLOTH_BENCH_JSON (\{.*\})", p.stdout)
    if not m:
        print(p.stdout[-3000:]); print(p.stderr[-3000:])
        raise RuntimeError(f"{arm} arm failed")
    res = json.loads(m.group(1))
    print(f"  -> {res['tokens_per_sec']:,.0f} tok/s | "
          f"{res['peak_reserved_gb']:.2f} GB reserved | loss {res['final_loss']:.4f}")
    return res

# Free the notebook's own model first — the subprocesses need the whole GPU.
for name in ("trainer", "model"):
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free before benchmark: {free/1e9:.2f} / {total/1e9:.2f} GB\n")

results = {
    "Unsloth (fused Triton)": run_arm("unsloth", steps=20, warmup=3),
    "Stock HF + peft + trl":  run_arm("hf",      steps=20, warmup=3),
}
Path("bench_results.json").write_text(json.dumps(results, indent=2))

In [ ]:
# ---- Results table --------------------------------------------------------------------
u = results["Unsloth (fused Triton)"]
h = results["Stock HF + peft + trl"]

def ratio(a, b):
    return f"{a/b:.2f}x" if b else "—"

print(f"{'metric':<26}{'Unsloth':>14}{'Stock HF':>14}{'delta':>14}")
print("-" * 68)
rows = [
    ("median step (s)",      u["median_step_s"],     h["median_step_s"],      "lower better"),
    ("tokens/sec",           u["tokens_per_sec"],    h["tokens_per_sec"],     "higher better"),
    ("peak allocated (GB)",  u["peak_alloc_gb"],     h["peak_alloc_gb"],      "lower better"),
    ("peak reserved (GB)",   u["peak_reserved_gb"],  h["peak_reserved_gb"],   "lower better"),
    ("VRAM after load (GB)", u["mem_after_load_gb"], h["mem_after_load_gb"],  "lower better"),
    ("final loss",           u["final_loss"],        h["final_loss"],         "should MATCH"),
]
for name, a, b, note in rows:
    print(f"{name:<26}{a:>14.4f}{b:>14.4f}{'':>4}{note}")

print(f"\nspeedup            : {ratio(u['tokens_per_sec'], h['tokens_per_sec'])}")
print(f"VRAM reduction     : {100*(1 - u['peak_reserved_gb']/h['peak_reserved_gb']):.1f}%")
print(f"loss agreement     : |Δ| = {abs(u['final_loss']-h['final_loss']):.4f}"
      f"   (fp16 noise; a large gap means the kernels are NOT equivalent)")
print(f"tokens forwarded   : {u['tokens_forwarded']:,} on both arms — equal by construction")

In [ ]:
# ---- Plot: two measures of different scale => two panels, never a dual axis -------------
import matplotlib.pyplot as plt

SURFACE, INK, INK_MUTED = "#fcfcfb", "#0b0b0b", "#52514e"
SERIES = {"Unsloth (fused Triton)": "#2a78d6", "Stock HF + peft + trl": "#eb6834"}
arms = list(SERIES)

panels = [
    ("Throughput", "tokens / sec",     [results[a]["tokens_per_sec"]   for a in arms], "higher is better"),
    ("Peak VRAM",  "GB reserved",      [results[a]["peak_reserved_gb"] for a in arms], "lower is better"),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), facecolor=SURFACE)
for ax, (title, unit, vals, hint) in zip(axes, panels):
    ax.set_facecolor(SURFACE)
    y = range(len(arms))
    bars = ax.barh(y, vals, height=0.55, color=[SERIES[a] for a in arms])
    # 4px-equivalent rounded data-ends would need a patch effect; keep marks thin and
    # let the direct labels carry the values.
    for b, v in zip(bars, vals):
        ax.text(b.get_width() * 1.02, b.get_y() + b.get_height() / 2,
                f"{v:,.0f}" if v > 100 else f"{v:.2f}",
                va="center", ha="left", color=INK, fontsize=10, fontweight="medium")
    ax.set_yticks([]); ax.set_xlim(0, max(vals) * 1.22)
    ax.set_title(f"{title}   ·   {unit}", loc="left", color=INK, fontsize=11, pad=10)
    ax.text(0, 1.0, hint, transform=ax.transAxes, ha="left", va="bottom",
            color=INK_MUTED, fontsize=9)
    ax.grid(axis="x", color="#e6e5e1", linewidth=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#d8d7d2")
    ax.tick_params(colors=INK_MUTED, labelsize=9, length=0)

# One legend for both panels — identity is never carried by color alone.
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in SERIES.values()]
fig.legend(handles, arms, loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.5, -0.09), fontsize=10, labelcolor=INK)
fig.suptitle(f"Fused Triton kernels vs. stock PyTorch — Llama-3.2-1B QLoRA on {GPU_NAME}",
             x=0.008, ha="left", color=INK, fontsize=12.5, fontweight="semibold")
fig.tight_layout(rect=(0, 0.02, 1, 0.92))
plt.savefig("unsloth_benchmark.png", dpi=160, bbox_inches="tight", facecolor=SURFACE)
plt.show()

---

## **[Key Observations]**

*Fill in from the runs above. A kernel result is only meaningful next to the configuration that produced it — copy the config row every time you quote a speedup.*

### Run configuration

| Setting | Value |
|---|---|
| GPU / compute capability | |
| unsloth / triton / transformers / torch versions | |
| Base model / vocab size | `unsloth/Llama-3.2-1B-Instruct` / 128,256 |
| `max_seq_length` / batch / accumulation | |
| LoRA `r` / `alpha` / target modules | |
| Gradient checkpointing (unsloth-offload vs stock) | |
| Benchmark steps / warmup excluded | |

### Correctness (Step 3) — fill this in *before* trusting any speed number

| Kernel | max abs Δ vs. reference | Verdict (fp16 eps = 9.8e-4) |
|---|---|---|
| cross-entropy value | | |
| cross-entropy `dlogits` | | |
| RMSNorm forward / `dX` | | |
| SwiGLU forward | | |

### Performance

| Metric | Unsloth | Stock HF | Ratio |
|---|---|---|---|
| Median step time (s) | | | |
| Tokens/sec | | | |
| Peak **allocated** (GB) | | | |
| Peak **reserved** (GB) | | | |
| VRAM after load (GB) | | | |
| Final loss | | | should match to fp16 noise |

### Things worth logging every time

- **Both** `max_memory_allocated` and `max_memory_reserved`. Reserved is what OOMs you.
- The **loss agreement** between arms. A speedup with a diverging loss is not a speedup.
- Whether the benchmark ran in **separate processes**. Same-process A/B is measuring Unsloth against Unsloth.
- **Where the win actually came from.** Re-run the Unsloth arm with `--ckpt true` to remove host-RAM activation offload: what remains is the pure kernel/gradient contribution. If most of the VRAM saving disappears, your win was offload, not fusion — a completely different thing to reason about at scale.
- **The maximum sequence length each arm survives.** Raise `--seq` until the stock arm OOMs; that crossover is the most honest one-number summary of this technique.

### Sensitivity sweeps worth running

- `--seq ∈ {512, 1024, 2048, 4096}` — the fused-CE advantage should grow roughly linearly, because the logit tensor does.
- A large-vocab model (Gemma-2, 256 k vocab) vs. a small-vocab one — isolates the CE contribution from everything else.
- Dropping `gate_proj/up_proj/down_proj` from `target_modules` — opts out of the fused `LoRA_MLP` backward and should visibly cost throughput, which is the cleanest attribution experiment in the notebook.

## Export — Download the Adapter, Benchmark and Plot (Optional)

> Ship the **benchmark JSON and the version table next to the weights.** A "2× faster" claim without the GPU, the sequence length, and the library versions is not a result, it is a rumour.

In [ ]:
import shutil

bundle = WORKDIR / "unsloth_bundle"
bundle.mkdir(exist_ok=True)
for src, dst in [("./outputs/lora_adapter", "lora_adapter")]:
    if Path(src).exists():
        shutil.copytree(src, bundle / dst, dirs_exist_ok=True)
for f in ("bench_results.json", "unsloth_benchmark.png", "bench_arm.py"):
    if Path(f).exists():
        shutil.copy(f, bundle / f)

(bundle / "VERSIONS.txt").write_text(
    f"gpu={GPU_NAME} sm_{SM}\ntorch={torch.__version__}\ntriton={triton.__version__}\n"
    f"transformers={transformers.__version__}\nunsloth={getattr(unsloth,'__version__','unknown')}\n"
)

output_filename = "unsloth_kernel_bundle.zip"
shutil.make_archive(output_filename.replace(".zip", ""), "zip", bundle)
print(f"File: {output_filename}  ({os.path.getsize(output_filename)/1e6:.2f} MB)")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found — run the export cell first.")